In [1]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

path = '/content/drive/MyDrive/capstone/'


import re

def tokenize(text, language):
  #print(1)
  tee = text.split('\n')
  tokens = []
  labels = []
  for i in tee:
    text_and_label = i.split(': ',1)
    if len(text_and_label) == 2:
      label = text_and_label[0]
      t = text_and_label[1]
    else:
      label = 'None'
      t = text_and_label[0]
    if language not in ['zh','ZH']:
      #tokenizes by spaces and punctuation
      tokens = tokens + t.split(' ')
      labels.append(label)
    else:
      ###adapted from chatgpt code
      t = re.sub(r'[^\w\u4e00-\u9fff]','',t)
      tokens = tokens + list(t)
      labels.append(label)
      ###
  tokens = [x for x in tokens if x != '']
  return tokens, labels
"""
def tokenize_line(line, language):
  text_and_label = line.split(': ',1)
  label = text_and_label[0]
  t = text_and_label[1]
  if language not in ['zh','ZH']:
    #tokenizes by spaces and punctuation
    tokens = t.split(' ')
    tokens = [x for x in tokens if x != '']
    return tokens, label
  else:
    ###adapted from chatgpt code
    t = re.sub(r'[^\w\u4e00-\u9fff]','',t)
    return list(t), label
"""

from collections import Counter
def find_intersect(pred, gold, language):
  p = tokenize(pred,language)[0]

  g = tokenize(gold,language)[0]
  #print(1)
  ###adapted from ChatGPT code
  g = Counter(g)

  intersect = []

  for token in p:
    if g[token] > 0:
      intersect.append(token)
      g[token] -= 1

  return intersect
  ###

def bag_of_words_score(intersect, tokenized_pred, tokenized_gold):
  total = tokenized_pred + tokenized_gold
  return 2*len(intersect)/len(total)

###GPT code###
from scipy.stats import spearmanr
def pos_by_gold_labels(intersect, gold):
  pos_map = {}
  for i, token in enumerate(gold):
    pos_map.setdefault(token, []).append(i)
  used = Counter()
  matched_positions = []

  for token in intersect:
    if token in pos_map and used[token] < len(pos_map[token]):
      matched_positions.append(pos_map[token][used[token]])
      used[token] += 1
  return matched_positions
###
def in_order_score(intersect, tokenized_gold):
  #output = bag_of_words(pred, gold, language)
  #g = tokenize(gold)[0]
  ###adapted from gpt code
  pos = pos_by_gold_labels(intersect, tokenized_gold)
  if len(pos) < 2:
    return 1.0
  ideal = sorted(pos)
  score, _ = spearmanr(pos, ideal)
  ###
  #lowest score is 0.5 so that even if stuff is out of order, the scores reflect that something was matched
  score = ((score + 1) / 4) + 0.5
  return score

def bag_complete_score(pred, gold, language):
  #print(2)
  intersect = find_intersect(pred, gold, language)
  #print(3)
  p = tokenize(pred, language)[0]
  g = tokenize(gold, language)[0]
  aa = bag_of_words_score(intersect, p, g)
  bb = in_order_score(intersect, g)

  return aa, bb, aa*bb

def exact_line_score(pred, gold):
  p = pred.split('\n')
  p = [x for x in p if x!='']
  g = gold.split('\n')
  g = [x for x in g if x!='']

  total_lines = len(p) + len(g)
  #print(total_lines)

  for i in range(len(p)):
    e = p[i].split(': ', 1)
    if len(e) == 2:
      p[i] = e
    else:
      p[i] = ['None', e[0]]
  for i in range(len(g)):
    e = g[i].split(': ', 1)
    if len(e) == 2:
      g[i] = e
    else:
      g[i] = ['None', e[0]]


  full_matches = 0
  i = 0
  while i<len(g):
    #see if there are label + text full match
    for j in range(len(p)):
      if g[i] == p[j]:
        full_matches += 1
        p.pop(j)
        g.pop(i)
        i -= 1
        break
    i += 1
  text_matches = 0
  i = 0
  while i<len(g):
    #see if there are text only match
    for j in range(len(p)):
      if g[i][1] == p[j][1]:
        text_matches += 1
        p.pop(j)
        g.pop(i)
        i -= 1
        break
    i += 1

  return 2*(text_matches*0.75 + full_matches)/total_lines
from statistics import mean
from math import isnan
def get_base_metrics(seed_start, seed_end, lang_train, lang_test, model_name, model_type, path = path):
  p = path + 'results/' + model_type + '/'

  aggregate_data = {'seed': [], 'bag_of_words_raw':[],'in_order':[],'bag_of_words_ordered':[],'line_score':[], 'average_score':[]}


  for i in range(seed_start, seed_end):
    a = pd.read_csv(p + model_name + '_train_' + lang_train + '_test_' + lang_test + '_' + str(i) + '.csv')

    bag_scores = []
    in_order_scores = []
    in_order_bag_scores = []
    line_scores = []
    average_scores = []

    #only take in the stuff where both prediction and gold label are not No MWE
    if 'compound' in model_name:
      a = a[a['output'] != 'No compound found.']
      a = a[a['label'] != 'No compound found.']
    else:
      a = a[a['output'] != 'No MWE found.']
      a = a[a['label'] != 'No MWE found.']

    for j in range(len(a['label'])):

      scores = bag_complete_score(a['output'].iloc[j],a['label'].iloc[j],lang_test)
      bag_scores.append(scores[0])
      in_order_scores.append(scores[1])
      in_order_bag_scores.append(scores[2])

      line_score = exact_line_score(a['output'].iloc[j],a['label'].iloc[j])
      line_scores.append(line_score)


      average_scores.append((scores[2]+line_score)/2)

    a['bag_of_words_raw'] = bag_scores
    a['in_order'] = in_order_scores
    a['bag_of_words_ordered'] = in_order_bag_scores
    a['line_score'] = line_scores
    a['average_score'] = average_scores
    a.to_csv(path + 'results/' + model_type + '-scored/' + model_name + '_train_' + lang_train + '_test_' + lang_test + '_' + str(i) + '.csv')

    aggregate_data['seed'].append(i)
    aggregate_data['bag_of_words_raw'].append(mean(bag_scores) if len(bag_scores) >= 1 else float('nan'))
    aggregate_data['in_order'].append(mean(in_order_scores) if len(in_order_scores) >= 1 else float('nan'))
    aggregate_data['bag_of_words_ordered'].append(mean(in_order_bag_scores) if len(in_order_bag_scores) >= 1 else float('nan'))
    aggregate_data['line_score'].append(mean(line_scores) if len(line_scores) >= 1 else float('nan'))
    aggregate_data['average_score'].append(mean(average_scores) if len(average_scores) >= 1 else float('nan'))

  aggregate_data['seed'].append('avg')
  y = [x for x in aggregate_data['bag_of_words_raw'] if not isnan(x)]
  aggregate_data['bag_of_words_raw'].append(mean(y) if len(y) >= 1 else float('nan'))
  y = [x for x in aggregate_data['in_order'] if not isnan(x)]
  aggregate_data['in_order'].append(mean(y) if len(y) >= 1 else float('nan'))
  y = [x for x in aggregate_data['bag_of_words_ordered'] if not isnan(x)]
  aggregate_data['bag_of_words_ordered'].append(mean(y) if len(y) >= 1 else float('nan'))
  y = [x for x in aggregate_data['line_score'] if not isnan(x)]
  aggregate_data['line_score'].append(mean(y) if len(y) >= 1 else float('nan'))
  y = [x for x in aggregate_data['average_score'] if not isnan(x)]
  aggregate_data['average_score'].append(mean(y) if len(y) >= 1 else float('nan'))

  aggregate_data = pd.DataFrame(aggregate_data)
  aggregate_data.to_csv(path + 'results/' + model_type + '-scored/' + model_name + '_train_' + lang_train + '_test_' + lang_test + '_summary.csv')


<>:47: SyntaxWarning: invalid escape sequence '\w'
<>:47: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_2251/977651237.py:47: SyntaxWarning: invalid escape sequence '\w'
  t = re.sub(r'[^\w\u4e00-\u9fff]','',t)


Mounted at /content/drive


In [ ]:
get_base_metrics(0, 5, 'en', 'EN', 'vmwe_0shot', 'llama-8b', path = path)

In [ ]:
get_base_metrics(0, 5, 'en', 'ES', 'vmwe_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'en', 'ZH', 'vmwe_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'en', 'TR', 'vmwe_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'en', 'EN', 'vmwe_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'en', 'ES', 'vmwe_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'en', 'ZH', 'vmwe_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'en', 'TR', 'vmwe_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'en', 'EN', 'compound_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'en', 'ES', 'compound_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'en', 'ZH', 'compound_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'en', 'TR', 'compound_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'en', 'EN', 'compound_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'en', 'ES', 'compound_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'en', 'ZH', 'compound_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'en', 'TR', 'compound_fewshot', 'llama-8b', path = path)

In [ ]:
get_base_metrics(0, 5, 'es', 'EN', 'vmwe_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'es', 'ES', 'vmwe_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'es', 'ZH', 'vmwe_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'es', 'TR', 'vmwe_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'es', 'EN', 'vmwe_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'es', 'ES', 'vmwe_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'es', 'ZH', 'vmwe_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'es', 'TR', 'vmwe_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'es', 'EN', 'compound_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'es', 'ES', 'compound_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'es', 'ZH', 'compound_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'es', 'TR', 'compound_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'es', 'EN', 'compound_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'es', 'ES', 'compound_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'es', 'ZH', 'compound_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'es', 'TR', 'compound_fewshot', 'llama-8b', path = path)

get_base_metrics(0, 5, 'zh', 'EN', 'vmwe_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'zh', 'ES', 'vmwe_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'zh', 'ZH', 'vmwe_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'zh', 'TR', 'vmwe_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'zh', 'EN', 'vmwe_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'zh', 'ES', 'vmwe_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'zh', 'ZH', 'vmwe_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'zh', 'TR', 'vmwe_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'zh', 'EN', 'compound_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'zh', 'ES', 'compound_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'zh', 'ZH', 'compound_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'zh', 'TR', 'compound_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'zh', 'EN', 'compound_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'zh', 'ES', 'compound_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'zh', 'ZH', 'compound_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'zh', 'TR', 'compound_fewshot', 'llama-8b', path = path)

get_base_metrics(0, 5, 'tr', 'EN', 'vmwe_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'tr', 'ES', 'vmwe_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'tr', 'ZH', 'vmwe_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'tr', 'TR', 'vmwe_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'tr', 'EN', 'vmwe_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'tr', 'ES', 'vmwe_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'tr', 'ZH', 'vmwe_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'tr', 'TR', 'vmwe_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'tr', 'EN', 'compound_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'tr', 'ES', 'compound_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'tr', 'ZH', 'compound_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'tr', 'TR', 'compound_0shot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'tr', 'EN', 'compound_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'tr', 'ES', 'compound_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'tr', 'ZH', 'compound_fewshot', 'llama-8b', path = path)
get_base_metrics(0, 5, 'tr', 'TR', 'compound_fewshot', 'llama-8b', path = path)

In [ ]:
get_base_metrics(0, 5, 'en', 'ES', 'vmwe_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'en', 'ZH', 'vmwe_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'en', 'TR', 'vmwe_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'en', 'EN', 'vmwe_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'en', 'ES', 'vmwe_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'en', 'ZH', 'vmwe_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'en', 'TR', 'vmwe_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'en', 'EN', 'compound_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'en', 'ES', 'compound_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'en', 'ZH', 'compound_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'en', 'TR', 'compound_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'en', 'EN', 'compound_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'en', 'ES', 'compound_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'en', 'ZH', 'compound_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'en', 'TR', 'compound_fewshot', 'qwen-32b', path = path)

get_base_metrics(0, 5, 'es', 'EN', 'vmwe_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'es', 'ES', 'vmwe_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'es', 'ZH', 'vmwe_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'es', 'TR', 'vmwe_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'es', 'EN', 'vmwe_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'es', 'ES', 'vmwe_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'es', 'ZH', 'vmwe_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'es', 'TR', 'vmwe_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'es', 'EN', 'compound_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'es', 'ES', 'compound_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'es', 'ZH', 'compound_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'es', 'TR', 'compound_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'es', 'EN', 'compound_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'es', 'ES', 'compound_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'es', 'ZH', 'compound_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'es', 'TR', 'compound_fewshot', 'qwen-32b', path = path)

get_base_metrics(0, 5, 'zh', 'EN', 'vmwe_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'zh', 'ES', 'vmwe_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'zh', 'ZH', 'vmwe_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'zh', 'TR', 'vmwe_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'zh', 'EN', 'vmwe_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'zh', 'ES', 'vmwe_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'zh', 'ZH', 'vmwe_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'zh', 'TR', 'vmwe_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'zh', 'EN', 'compound_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'zh', 'ES', 'compound_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'zh', 'ZH', 'compound_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'zh', 'TR', 'compound_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'zh', 'EN', 'compound_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'zh', 'ES', 'compound_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'zh', 'ZH', 'compound_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'zh', 'TR', 'compound_fewshot', 'qwen-32b', path = path)

get_base_metrics(0, 5, 'tr', 'EN', 'vmwe_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'tr', 'ES', 'vmwe_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'tr', 'ZH', 'vmwe_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'tr', 'TR', 'vmwe_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'tr', 'EN', 'vmwe_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'tr', 'ES', 'vmwe_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'tr', 'ZH', 'vmwe_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'tr', 'TR', 'vmwe_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'tr', 'EN', 'compound_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'tr', 'ES', 'compound_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'tr', 'ZH', 'compound_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'tr', 'TR', 'compound_0shot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'tr', 'EN', 'compound_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'tr', 'ES', 'compound_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'tr', 'ZH', 'compound_fewshot', 'qwen-32b', path = path)
get_base_metrics(0, 5, 'tr', 'TR', 'compound_fewshot', 'qwen-32b', path = path)

In [2]:
get_base_metrics(0, 1, 'na', 'EN', 'vmwe_0shot', 'qwen-32b-base', path = path)
get_base_metrics(0, 1, 'na', 'ES', 'vmwe_0shot', 'qwen-32b-base', path = path)
get_base_metrics(0, 1, 'na', 'ZH', 'vmwe_0shot', 'qwen-32b-base', path = path)
get_base_metrics(0, 1, 'na', 'TR', 'vmwe_0shot', 'qwen-32b-base', path = path)
get_base_metrics(0, 1, 'na', 'EN', 'vmwe_fewshot', 'qwen-32b-base', path = path)
get_base_metrics(0, 1, 'na', 'ES', 'vmwe_fewshot', 'qwen-32b-base', path = path)
get_base_metrics(0, 1, 'na', 'ZH', 'vmwe_fewshot', 'qwen-32b-base', path = path)
get_base_metrics(0, 1, 'na', 'TR', 'vmwe_fewshot', 'qwen-32b-base', path = path)
get_base_metrics(0, 1, 'na', 'EN', 'compound_0shot', 'qwen-32b-base', path = path)
get_base_metrics(0, 1, 'na', 'ES', 'compound_0shot', 'qwen-32b-base', path = path)
get_base_metrics(0, 1, 'na', 'ZH', 'compound_0shot', 'qwen-32b-base', path = path)
get_base_metrics(0, 1, 'na', 'TR', 'compound_0shot', 'qwen-32b-base', path = path)
get_base_metrics(0, 1, 'na', 'EN', 'compound_fewshot', 'qwen-32b-base', path = path)
get_base_metrics(0, 1, 'na', 'ES', 'compound_fewshot', 'qwen-32b-base', path = path)
get_base_metrics(0, 1, 'na', 'ZH', 'compound_fewshot', 'qwen-32b-base', path = path)
get_base_metrics(0, 1, 'na', 'TR', 'compound_fewshot', 'qwen-32b-base', path = path)

get_base_metrics(0, 1, 'na', 'EN', 'vmwe_0shot', 'llama-8b-base', path = path)
get_base_metrics(0, 1, 'na', 'ES', 'vmwe_0shot', 'llama-8b-base', path = path)
get_base_metrics(0, 1, 'na', 'ZH', 'vmwe_0shot', 'llama-8b-base', path = path)
get_base_metrics(0, 1, 'na', 'TR', 'vmwe_0shot', 'llama-8b-base', path = path)
get_base_metrics(0, 1, 'na', 'EN', 'vmwe_fewshot', 'llama-8b-base', path = path)
get_base_metrics(0, 1, 'na', 'ES', 'vmwe_fewshot', 'llama-8b-base', path = path)
get_base_metrics(0, 1, 'na', 'ZH', 'vmwe_fewshot', 'llama-8b-base', path = path)
get_base_metrics(0, 1, 'na', 'TR', 'vmwe_fewshot', 'llama-8b-base', path = path)
get_base_metrics(0, 1, 'na', 'EN', 'compound_0shot', 'llama-8b-base', path = path)
get_base_metrics(0, 1, 'na', 'ES', 'compound_0shot', 'llama-8b-base', path = path)
get_base_metrics(0, 1, 'na', 'ZH', 'compound_0shot', 'llama-8b-base', path = path)
get_base_metrics(0, 1, 'na', 'TR', 'compound_0shot', 'llama-8b-base', path = path)
get_base_metrics(0, 1, 'na', 'EN', 'compound_fewshot', 'llama-8b-base', path = path)
get_base_metrics(0, 1, 'na', 'ES', 'compound_fewshot', 'llama-8b-base', path = path)
get_base_metrics(0, 1, 'na', 'ZH', 'compound_fewshot', 'llama-8b-base', path = path)
get_base_metrics(0, 1, 'na', 'TR', 'compound_fewshot', 'llama-8b-base', path = path)

In [4]:
test = ['EN','ES','ZH','TR']
model_names = ['vmwe_0shot', 'vmwe_fewshot', 'compound_0shot', 'compound_fewshot']
def get_average_table(lang_train, model_type,  test = test, model_names = model_names, path = path):
  p = path + 'results/' + model_type + '-scored/'
  results = {'model': [], 'setup': [], 'train': [], 'test': [], 'bag_of_words_score': [], 'exact_line_score': [], 'average_score': []}
  for lang_test in test:
    for model_name in model_names:
      a = pd.read_csv(p + model_name + '_train_' + lang_train + '_test_' + lang_test + '_summary.csv')
      a = a[a['seed']=='avg']#should be only one line per file

      results['model'].append(model_type)
      results['setup'].append(model_name)
      results['train'].append(lang_train)
      results['test'].append(lang_test)
      results['bag_of_words_score'].append(a.iloc[0]['bag_of_words_ordered'])
      results['exact_line_score'].append(a.iloc[0]['line_score'])
      results['average_score'].append(a.iloc[0]['average_score'])
  results = pd.DataFrame(results)
  results.to_csv(p+lang_train+'_averages.csv')

In [11]:
#get_base_metrics(0, 5, 'en', 'EN', 'vmwe_0shot', 'qwen-32b', path = path)
get_average_table('na','llama-8b-base')
get_average_table('na','qwen-32b-base')